# 04 — Pipeline ASR đầy đủ: PhoWhisper-medium → LoRA → chọn theo số câu sai → model merge

**Kaggle:** Accelerator = **GPU T4** (hoặc P100), Internet = **On** (tải model/adapter từ Hugging Face), Persistence = Files.
**Input bắt buộc:** dataset private chứa thư mục `train+public_test` của BTC (notebook tự tìm `train/train.jsonl`).
**Input tùy chọn:** dataset chứa thư mục `PhoWhisper-medium` (có `config.json`), dùng khi không muốn tải từ HF.

Luồng: cấu hình → data + split dev cố định (309 hội thoại, giữ nguyên split của team) → baseline (gốc / gốc+adapter) → chọn điểm xuất phát
→ tạo nhãn số → LoRA r=32 enc+dec + SpecAugment + augmentation → eval dev mỗi epoch, **chọn checkpoint theo số câu sai** (phụ: WER), early stop
→ so greedy vs beam → merge fp16 → kiểm tra merge + latency → phân tích lỗi → đóng gói output.

Chỉ số chính: **`n_wrong`** = số câu dev sai sau chuẩn hóa (chữ thường, bỏ dấu câu, số đọc thành chữ, chấp nhận mọi cách đọc số hợp lệ).
Chạy thử nhanh: đặt `SMOKE = True`.

In [ ]:
# ===== Cấu hình =====
SMOKE = False            # True: vài chục câu, 1 epoch — chỉ để kiểm tra pipeline chạy thông
BASE_ID = "vinai/PhoWhisper-medium"
ADAPTER_ID = "schaffen49/PhoWhisper_medium_lora"   # "" = bỏ qua adapter có sẵn
START = "auto"           # "auto" (chọn cái ít câu sai hơn trên dev) | "base" | "adapter"
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]   # cả encoder và decoder
EPOCHS = 10
PATIENCE = 3             # dừng nếu n_wrong không giảm sau 3 epoch
LR = 1e-4
BATCH = 8
GRAD_ACC = 2
WARMUP = 0.1
WEIGHT_DECAY = 0.01
AUG_P = 0.3              # xác suất augment audio mỗi câu train
SPEC_AUG = True          # SpecAugment của Whisper (chỉ bật khi train)
EVAL_BATCH = 16
MAX_NEW_TOKENS = 128
FINAL_BEAMS = [1, 5]     # so greedy vs beam ở bước cuối; beam tốt hơn được ghi vào generation_config
SEED = 42
PRINT_WRONG = 30         # in tối đa bao nhiêu câu sai sau mỗi lần chấm
RESUME = False           # True: bỏ qua baseline + train, dùng best_lora.pt/results.json/history.json của lần chạy trước (add output đó làm Input)
import os
DATA_ROOTS = [p for p in [os.environ.get("ASR_DATA_ROOT"), "/kaggle/input"] if p]
OUT = os.environ.get("ASR_OUT", "/kaggle/working/asr_pipeline")

In [ ]:
!pip uninstall -y -q torchao
!pip install -q "peft>=0.13,<1"

In [ ]:
import os, sys, json, time, math, random, wave, glob, csv, shutil
from pathlib import Path
import numpy as np, torch
import transformers
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)
try:
    import peft; print("peft", peft.__version__)
except ImportError:
    print("peft chưa cài (chỉ cần cho adapter/train)")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
os.makedirs(OUT, exist_ok=True)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
DEV = "cuda"

In [ ]:
# ===== Chuẩn hóa văn bản + đọc số tiếng Việt (đã unit-test) =====
# ===== Chuẩn hóa văn bản + đọc số tiếng Việt =====
import re, unicodedata, itertools
from functools import lru_cache

D = ["không", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]
# Mỗi cờ: (giá trị mặc định, biến thể)
FLAG_CHOICES = {
    "drop_muoi": (False, True),     # "hai mươi ba" / "hai ba"
    "tu": ("bốn", "tư"),            # 24: "hai mươi bốn" / "hai mươi tư"
    "lam": ("lăm", "nhăm"),         # 25: "lăm" / "nhăm"
    "linh": ("linh", "lẻ"),         # 105: "một trăm linh năm" / "lẻ năm"
    "nghin": ("nghìn", "ngàn"),
    "full": (False, True),          # 1.050.000: có/không "không trăm"
    "ty": ("tỷ", "tỉ"),
}
FLAG_NAMES = list(FLAG_CHOICES)
DEFAULT_FLAGS = tuple(FLAG_CHOICES[k][0] for k in FLAG_NAMES)
ALL_FLAGS = list(itertools.product(*(FLAG_CHOICES[k] for k in FLAG_NAMES)))


def _triple(g, f, lower, full):
    h, t, u = g // 100, g // 10 % 10, g % 10
    w = []
    if h > 0 or (full and lower):
        w += [D[h], "trăm"]
    said_h = bool(w)
    if t == 0:
        if u:
            w += ([f["linh"]] if (said_h or lower) else []) + [D[u]]
    elif t == 1:
        w += ["mười"] + (["lăm"] if u == 5 else [D[u]] if u else [])
    else:
        w += [D[t]] + ([] if (f["drop_muoi"] and u) else ["mươi"])
        w += {0: [], 1: ["mốt"], 4: [f["tu"]], 5: [f["lam"]]}.get(u, [D[u]] if u else [])
    return w


def read_int(n, flags=DEFAULT_FLAGS):
    f = dict(zip(FLAG_NAMES, flags))
    if n == 0:
        return "không"
    groups = []
    while n:
        groups.append(n % 1000)
        n //= 1000
    units = ["", f["nghin"], "triệu", f["ty"]]
    w = []
    top = len(groups) - 1
    for i in range(top, -1, -1):
        g = groups[i]
        if g == 0:
            continue
        w += _triple(g, f, lower=(i < top), full=f["full"])
        if units[i]:
            w.append(units[i])
    return " ".join(w)


def read_digits(s):
    return " ".join(D[int(c)] for c in s)


@lru_cache(maxsize=None)
def number_candidates(tok):
    """Mọi cách đọc hợp lệ của một chuỗi số; phần tử đầu là cách đọc mặc định."""
    out = []
    if re.fullmatch(r"\d{1,3}(\.\d{3})+", tok):            # 50.000 -> tách nghìn
        tok = tok.replace(".", "")
    m = re.fullmatch(r"(\d+)[.,](\d+)", tok)                # thập phân
    if m:
        a, b = m.groups()
        for ia in number_candidates(a):
            for sep in ("phẩy", "chấm"):
                out.append(f"{ia} {sep} {read_digits(b) if len(b) > 1 else D[int(b)]}")
            if b == "5":
                out.append(f"{ia} rưỡi")
        return tuple(dict.fromkeys(out))
    if len(tok) > 1 and tok.startswith("0") or len(tok) > 12:   # số điện thoại, mã
        return (read_digits(tok),)
    n = int(tok)
    out = [read_int(n, fl) for fl in [DEFAULT_FLAGS] + ALL_FLAGS]
    if len(tok) > 1:
        out.append(read_digits(tok))                        # đọc lẻ: RV107, vé 412750
    return tuple(dict.fromkeys(out))


NUM_RE = re.compile(r"\d+(?:[.,]\d+)*")
_OLD_TONE = {"oà": "òa", "oá": "óa", "oả": "ỏa", "oã": "õa", "oạ": "ọa",
             "oè": "òe", "oé": "óe", "oẻ": "ỏe", "oẽ": "õe", "oẹ": "ọe",
             "uỳ": "ùy", "uý": "úy", "uỷ": "ủy", "uỹ": "ũy", "uỵ": "ụy"}
_OLD_TONE_RE = re.compile(r"(?<!q)(" + "|".join(_OLD_TONE) + r")(?![a-zà-ỹ])")
EQUIV = {"ngàn": "nghìn", "tỉ": "tỷ", "nhăm": "lăm", "lẻ": "linh"}


def basic_norm(s):
    s = unicodedata.normalize("NFC", s).lower()
    s = s.replace("%", " phần trăm ")
    s = re.sub(r"[^\w\s]", " ", s)          # bỏ dấu câu, gạch nối
    s = s.replace("_", " ")
    return re.sub(r"\s+", " ", s).strip()


def _spans(s):
    """Tách số khỏi chữ dính liền (RON95 -> RON 95), giữ dấu . , bên trong số."""
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"(?<=[^\W\d_])(?=\d)|(?<=\d)(?=[^\W\d_])", " ", s)
    s = re.sub(r"[.,](?!\d)", " ", s)       # dấu . , cuối câu/không nằm giữa số
    return s


def cer(a, b):
    return edit_distance(list(a), list(b)) / max(1, len(a))


def verbalize(text, hyp=None):
    """Chuyển chữ số -> chữ. Có hyp (đã chuẩn hóa): mỗi số chọn cách đọc khớp hyp nhất."""
    s = _spans(text)
    parts = NUM_RE.split(s)
    nums = NUM_RE.findall(s)
    chosen = [number_candidates(n)[0] for n in nums]
    if hyp is not None and nums:
        for i, n in enumerate(nums):
            best, best_d = chosen[i], None
            for c in number_candidates(n):
                trial = chosen[:i] + [c] + chosen[i + 1:]
                cand = basic_norm("".join(p + (f" {trial[j]} " if j < len(trial) else "")
                                          for j, p in enumerate(parts)))
                d = cer(cand, hyp)
                if best_d is None or d < best_d:
                    best, best_d = c, d
            chosen[i] = best
    out = "".join(p + (f" {chosen[j]} " if j < len(chosen) else "") for j, p in enumerate(parts))
    return basic_norm(out)


def to_label(text, hyp=None):
    """Nhãn train: chữ thường, NFC, bỏ dấu câu, số viết bằng chữ."""
    return verbalize(text, hyp)


def eval_norm(s):
    """Chuẩn hóa thêm khi CHẤM (không dùng cho nhãn train): gộp biến thể tương đương."""
    s = _OLD_TONE_RE.sub(lambda m: _OLD_TONE[m.group(1)], basic_norm(s))
    return " ".join(EQUIV.get(w, w) for w in s.split())


def edit_distance(r, h):
    prev = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        cur = [i] + [0] * len(h)
        ri = r[i - 1]
        for j in range(1, len(h) + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ri != h[j - 1]))
        prev = cur
    return prev[-1]


NUM_WORDS = set(D) | {"mười", "mươi", "trăm", "nghìn", "triệu", "tỷ", "linh", "mốt", "lăm", "tư", "rưỡi", "kém", "phẩy"}
NEG_WORDS = ["đừng", "không", "thôi", "khỏi", "hủy", "bỏ"]


def score(refs_raw, hyps_raw):
    """refs_raw: nhãn gốc BTC (có thể chứa chữ số). Với mỗi hyp, số trong ref được đọc
    theo cách khớp hyp nhất (mọi cách đọc hợp lệ đều được chấp nhận)."""
    we = wn = ce = cn = exact = 0
    num_tot = num_ok = 0
    neg_ref = neg_hit = 0
    refs_used = []
    for r_raw, h_raw in zip(refs_raw, hyps_raw):
        h = eval_norm(verbalize(h_raw))
        r = eval_norm(verbalize(r_raw, hyp=h))
        refs_used.append(r)
        rw, hw = r.split(), h.split()
        we += edit_distance(rw, hw); wn += len(rw)
        ce += edit_distance(list(r), list(h)); cn += len(r)
        exact += r == h
        rn = [w for w in rw if w in NUM_WORDS]
        if rn:
            num_tot += 1
            num_ok += rn == [w for w in hw if w in NUM_WORDS]
        for k in NEG_WORDS:
            c = rw.count(k)
            neg_ref += c
            neg_hit += min(c, hw.count(k))
    n = max(1, len(refs_raw))
    return {"n": len(refs_raw), "WER": we / max(1, wn), "CER": ce / max(1, cn),
            "sent_exact": exact / n,
            "num_seq_acc": num_ok / max(1, num_tot), "n_num_utts": num_tot,
            "neg_recall": neg_hit / max(1, neg_ref)}, refs_used

# kiểm tra nhanh
assert read_int(1050000) == "một triệu năm mươi nghìn"
assert to_label("Xăng RON95", hyp="xăng rôn chín lăm") == "xăng ron chín lăm"
assert to_label("gọi 0912") == "gọi không chín một hai"
print("text utils OK")

In [ ]:
DEV_IDS = ["train_0011", "train_0015", "train_0028", "train_0033", "train_0041", "train_0042", "train_0045", "train_0068", "train_0072", "train_0074", "train_0076", "train_0078", "train_0085", "train_0089", "train_0098", "train_0108", "train_0112", "train_0123", "train_0127", "train_0131", "train_0132", "train_0139", "train_0140", "train_0141", "train_0157", "train_0177", "train_0181", "train_0189", "train_0193", "train_0195", "train_0201", "train_0204", "train_0206", "train_0207", "train_0212", "train_0215", "train_0225", "train_0228", "train_0245", "train_0248", "train_0250", "train_0251", "train_0252", "train_0254", "train_0257", "train_0265", "train_0266", "train_0280", "train_0283", "train_0284", "train_0295", "train_0301", "train_0306", "train_0314", "train_0317", "train_0322", "train_0327", "train_0330", "train_0332", "train_0336", "train_0353", "train_0359", "train_0363", "train_0365", "train_0377", "train_0379", "train_0387", "train_0388", "train_0389", "train_0390", "train_0398", "train_0400", "train_0401", "train_0404", "train_0405", "train_0411", "train_0417", "train_0423", "train_0433", "train_0434", "train_0447", "train_0451", "train_0456", "train_0462", "train_0486", "train_0508", "train_0511", "train_0513", "train_0527", "train_0532", "train_0533", "train_0539", "train_0540", "train_0562", "train_0575", "train_0577", "train_0579", "train_0583", "train_0588", "train_0596", "train_0599", "train_0600", "train_0629", "train_0641", "train_0645", "train_0657", "train_0660", "train_0664", "train_0674", "train_0675", "train_0677", "train_0685", "train_0686", "train_0691", "train_0696", "train_0704", "train_0712", "train_0716", "train_0721", "train_0725", "train_0732", "train_0743", "train_0744", "train_0770", "train_0772", "train_0776", "train_0777", "train_0784", "train_0791", "train_0794", "train_0797", "train_0802", "train_0810", "train_0820", "train_0829", "train_0836", "train_0837", "train_0838", "train_0840", "train_0857", "train_0866", "train_0871", "train_0878", "train_0879", "train_0885", "train_0887", "train_0888", "train_0890", "train_0892", "train_0893", "train_0900", "train_0902", "train_0908", "train_0909", "train_0910", "train_0920", "train_0921", "train_0929", "train_0933", "train_0940", "train_0943", "train_0946", "train_0951", "train_0952", "train_0964", "train_0966", "train_0977", "train_0981", "train_0983", "train_0986", "train_1006", "train_1020", "train_1023", "train_1029", "train_1056", "train_1062", "train_1066", "train_1067", "train_1074", "train_1076", "train_1077", "train_1081", "train_1085", "train_1088", "train_1090", "train_1093", "train_1096", "train_1107", "train_1113", "train_1121", "train_1131", "train_1135", "train_1137", "train_1147", "train_1161", "train_1162", "train_1172", "train_1173", "train_1181", "train_1192", "train_1197", "train_1200", "train_1203", "train_1208", "train_1216", "train_1219", "train_1221", "train_1224", "train_1227", "train_1228", "train_1229", "train_1236", "train_1241", "train_1242", "train_1248", "train_1250", "train_1252", "train_1255", "train_1264", "train_1268", "train_1270", "train_1274", "train_1279", "train_1286", "train_1305", "train_1308", "train_1311", "train_1315", "train_1322", "train_1329", "train_1335", "train_1339", "train_1354", "train_1356", "train_1361", "train_1368", "train_1370", "train_1371", "train_1372", "train_1380", "train_1388", "train_1394", "train_1396", "train_1399", "train_1401", "train_1403", "train_1405", "train_1409", "train_1411", "train_1413", "train_1415", "train_1424", "train_1426", "train_1427", "train_1431", "train_1434", "train_1436", "train_1437", "train_1452", "train_1454", "train_1462", "train_1463", "train_1464", "train_1471", "train_1478", "train_1481", "train_1486", "train_1489", "train_1506", "train_1514", "train_1519", "train_1540", "train_1544", "train_1545", "train_1551", "train_1555", "train_1557", "train_1558", "train_1561", "train_1565", "train_1577", "train_1588", "train_1591", "train_1592", "train_1595", "train_1600", "train_1617", "train_1623", "train_1629", "train_1632", "train_1635", "train_1643", "train_1646", "train_1648", "train_1653", "train_1664", "train_1673", "train_1675", "train_1676", "train_1688", "train_1695", "train_1696", "train_1704", "train_1713", "train_1716", "train_1721", "train_1722", "train_1738", "train_1740"]  # = reports/data_audit/dev_ids.json (seed 42, 309 hội thoại)

# ===== Data =====
def find_one(pattern):
    for root in DATA_ROOTS:
        hits = sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True))
        if hits:
            return hits[0]
    return None

train_jsonl = find_one("train/train.jsonl")
assert train_jsonl, f"Không thấy train/train.jsonl trong {DATA_ROOTS}"
SPLIT_DIR = Path(train_jsonl).parent
rows = [json.loads(l) for l in open(train_jsonl, encoding="utf-8")]
dev_ids = set(DEV_IDS)
assert dev_ids <= {r["id"] for r in rows}, "dev_ids không khớp train.jsonl — sai phiên bản data?"

def utterances(r):
    out = [{"sid": r["id"], "turn": "final", "audio": str(SPLIT_DIR / r["audio"]), "text": r["audio_text"]}]
    k = 0
    for t in r["history"]:
        if t.get("role") == "user" and "audio" in t:
            k += 1
            out.append({"sid": r["id"], "turn": f"u{k}", "audio": str(SPLIT_DIR / t["audio"]), "text": t["audio_text"]})
    return out

train_utts = [u for r in rows if r["id"] not in dev_ids for u in utterances(r)]
dev_utts = [u for r in rows if r["id"] in dev_ids for u in utterances(r)]
if SMOKE:
    train_utts, dev_utts = train_utts[:48], dev_utts[:24]
print(f"data: {SPLIT_DIR}\nsamples {len(rows)} | dev samples {len(dev_ids)} | train utts {len(train_utts)} | dev utts {len(dev_utts)}")

def load_wav(path):
    with wave.open(path) as a:
        sr, ch, sw = a.getframerate(), a.getnchannels(), a.getsampwidth()
        x = np.frombuffer(a.readframes(a.getnframes()), dtype="<i2").astype(np.float32) / 32768.0
    if ch > 1: x = x.reshape(-1, ch).mean(1)
    if sr != 16000 or sw != 2:
        import librosa; x = librosa.resample(x, orig_sr=sr, target_sr=16000)
    return x[: 30 * 16000]

for u in train_utts + dev_utts:
    with wave.open(u["audio"]) as a: u["dur"] = a.getnframes() / a.getframerate()
for name, us in [("train", train_utts), ("dev", dev_utts)]:
    d = np.array([u["dur"] for u in us])
    print(f"{name}: {d.sum()/3600:.2f} h | median {np.median(d):.1f}s | max {d.max():.1f}s | >30s: {(d > 30).sum()}")

In [ ]:
# ===== Model =====
from transformers import WhisperForConditionalGeneration, WhisperProcessor

local_cfg = find_one("PhoWhisper-medium/config.json")
MODEL_SRC = str(Path(local_cfg).parent) if local_cfg else BASE_ID
print("base model:", MODEL_SRC)
processor = WhisperProcessor.from_pretrained(MODEL_SRC)
tok = processor.tokenizer
tok.set_prefix_tokens(language="vietnamese", task="transcribe")

def load_base(dtype=torch.float32):
    try:
        m = WhisperForConditionalGeneration.from_pretrained(MODEL_SRC, dtype=dtype)
    except TypeError:
        m = WhisperForConditionalGeneration.from_pretrained(MODEL_SRC, torch_dtype=dtype)
    m.generation_config.forced_decoder_ids = None
    m.config.forced_decoder_ids = None
    return m

ADAPTER = None   # (config, state_dict) của adapter có sẵn
if ADAPTER_ID and START != "base":
    try:
        from huggingface_hub import hf_hub_download
        from safetensors.torch import load_file
        cfg = json.load(open(hf_hub_download(ADAPTER_ID, "adapter_config.json")))
        ADAPTER = (cfg, load_file(hf_hub_download(ADAPTER_ID, "adapter_model.safetensors")))
        print("adapter:", {k: cfg[k] for k in ["r", "lora_alpha", "target_modules"]}, "| tensors", len(ADAPTER[1]))
    except Exception as e:
        print("Không tải được adapter -> bắt đầu từ model gốc:", repr(e)[:200])
        START = "base"

def with_adapter(m):
    # Gắn adapter có sẵn rồi merge vào trọng số -> trả về WhisperForConditionalGeneration thường.
    from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
    cfg, sd = ADAPTER
    pm = get_peft_model(m, LoraConfig(r=cfg["r"], lora_alpha=cfg["lora_alpha"], lora_dropout=0.0,
                                      target_modules=cfg["target_modules"], bias="none"))
    res = set_peft_model_state_dict(pm, sd)
    missing = [k for k in getattr(res, "missing_keys", []) if "lora_" in k]
    unexpected = getattr(res, "unexpected_keys", [])
    assert not missing and not unexpected, f"Adapter không khớp: missing {len(missing)} unexpected {len(unexpected)}"
    return pm.merge_and_unload()

In [ ]:
# ===== Transcribe + chấm =====
@torch.no_grad()
def transcribe(model, utts, bs=EVAL_BATCH, beams=1, desc=""):
    model.eval()
    order = sorted(range(len(utts)), key=lambda i: utts[i]["dur"])
    hyps = [None] * len(utts)
    t0 = time.time()
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        feats = processor.feature_extractor([load_wav(utts[i]["audio"]) for i in idx],
                                            sampling_rate=16000, return_tensors="pt").input_features
        with torch.autocast("cuda", dtype=torch.float16):
            out = model.generate(input_features=feats.to(DEV), language="vi", task="transcribe",
                                 do_sample=False, num_beams=beams, max_new_tokens=MAX_NEW_TOKENS, use_cache=True)
        for i, t in zip(idx, tok.batch_decode(out, skip_special_tokens=True)):
            hyps[i] = t.strip()
    dt = time.time() - t0
    print(f"{desc}: {len(utts)} utts in {dt:.0f}s ({1000*dt/len(utts):.0f} ms/utt, batch {bs}, beams {beams})")
    return hyps

def evaluate(name, hyps, utts=None, show=PRINT_WRONG):
    utts = utts or dev_utts
    m, refs = score([u["text"] for u in utts], hyps)
    hn = [eval_norm(verbalize(h)) for h in hyps]
    wrong = [i for i in range(len(utts)) if refs[i] != hn[i]]
    m["n_wrong"] = len(wrong)
    fin = [i for i, u in enumerate(utts) if u["turn"] == "final"]
    m["n_final"] = len(fin)
    m["n_wrong_final"] = sum(refs[i] != hn[i] for i in fin)
    m["WER_final_turn"] = score([utts[i]["text"] for i in fin], [hyps[i] for i in fin])[0]["WER"]
    with open(f"{OUT}/dev_{name}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["sid", "turn", "ok", "ref_raw", "ref_norm", "hyp", "hyp_norm"])
        for u, r, h, n in zip(utts, refs, hyps, hn):
            w.writerow([u["sid"], u["turn"], int(r == n), u["text"], r, h, n])
    for i in wrong[:show]:
        print(f"  [ERR] {utts[i]['sid']}/{utts[i]['turn']}\n     REF: {refs[i]}\n     HYP: {hn[i]}")
    print(f"== {name}: n_wrong {m['n_wrong']}/{m['n']} (final turn {m['n_wrong_final']}/{m['n_final']}) | "
          + " ".join(f"{k} {m[k]:.4f}" for k in ["WER", "CER", "WER_final_turn", "num_seq_acc", "neg_recall"]))
    return m

def better(a, b):
    # Tiêu chí chọn: ít câu sai hơn; hòa thì WER thấp hơn.
    return (a["n_wrong"], a["WER"]) < (b["n_wrong"], b["WER"])

RESULTS = {}

In [ ]:
# ===== Baseline + chọn điểm xuất phát =====
RESUME_DIR = None
if RESUME:
    hit = find_one("asr_pipeline/best_lora.pt")
    assert hit, "RESUME=True nhưng không thấy asr_pipeline/best_lora.pt trong Input"
    RESUME_DIR = str(Path(hit).parent)
    old = json.load(open(f"{RESUME_DIR}/results.json", encoding="utf-8"))
    RESULTS.update({k: old[k] for k in ("base", "adapter") if k in old})
    print("RESUME từ", RESUME_DIR, "| baseline lấy từ results.json cũ")
else:
    m = load_base(torch.float16).to(DEV)
    RESULTS["base"] = evaluate("base", transcribe(m, dev_utts, desc="base"))
    del m; torch.cuda.empty_cache()

if ADAPTER is not None and not RESUME:
    m = with_adapter(load_base(torch.float16)).to(DEV)
    RESULTS["adapter"] = evaluate("adapter", transcribe(m, dev_utts, desc="base+adapter"))
    del m; torch.cuda.empty_cache()

if START == "auto":
    START = "adapter" if "adapter" in RESULTS and better(RESULTS["adapter"], RESULTS["base"]) else "base"
print("Điểm xuất phát:", START, "| n_wrong", RESULTS[START]["n_wrong"])
json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=1, ensure_ascii=False)

def load_start(dtype):
    m = load_base(dtype)
    return with_adapter(m) if START == "adapter" else m

In [ ]:
# ===== Nhãn train: số viết bằng chữ, chọn cách đọc khớp audio nhất =====
need = [u for u in train_utts if re.search(r"\d", u["text"])]
if need and not RESUME:
    m = load_start(torch.float16).to(DEV)
    for u, h in zip(need, transcribe(m, need, desc="train utts có chữ số")):
        u["asr_hint"] = eval_norm(verbalize(h))
    del m; torch.cuda.empty_cache()
for u in train_utts:
    u["label"] = to_label(u["text"], hyp=u.get("asr_hint"))
with open(f"{OUT}/train_labels_numbers.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["audio", "raw", "asr_hint", "label"])
    for u in need: w.writerow([u["audio"], u["text"], u.get("asr_hint", ""), u["label"]])   # RESUME: không có asr_hint
print(f"{len(need)} câu có chữ số; ví dụ:", [(u["text"], u["label"]) for u in need[:3]])

In [ ]:
# ===== Dataset + augmentation =====
from scipy.signal import resample_poly
def augment(x, rng):
    if rng.random() < 0.5:                          # băng thông điện thoại 8 kHz
        x = resample_poly(resample_poly(x, 1, 2), 2, 1)[: len(x)]
    if rng.random() < 0.6:                          # nhiễu trắng SNR 10–30 dB
        snr = rng.uniform(10, 30); p = np.mean(x ** 2) + 1e-9
        x = x + rng.normal(0, np.sqrt(p / 10 ** (snr / 10)), len(x)).astype(np.float32)
    if rng.random() < 0.3:                          # đổi tốc độ 0.9–1.1 (đổi cả cao độ)
        r = rng.choice([9, 10, 11]); x = resample_poly(x, 10, r) if r != 10 else x
    if rng.random() < 0.5:                          # đổi gain
        x = x * rng.uniform(0.5, 1.5)
    return np.clip(x, -1, 1).astype(np.float32)

class UttDS(torch.utils.data.Dataset):
    def __init__(self, utts, aug_p): self.u, self.aug_p, self.epoch = utts, aug_p, 0
    def __len__(self): return len(self.u)
    def __getitem__(self, i):
        u = self.u[i]; x = load_wav(u["audio"])
        rng = np.random.default_rng([SEED, self.epoch, i])     # tái lập được
        if self.aug_p > 0 and rng.random() < self.aug_p: x = augment(x, rng)
        feat = processor.feature_extractor(x[: 30 * 16000], sampling_rate=16000, return_tensors="np").input_features[0]
        return feat, tok(u["label"]).input_ids[:440]

SOT = tok.convert_tokens_to_ids("<|startoftranscript|>")
def collate(batch):
    feats = torch.tensor(np.stack([b[0] for b in batch]))
    ids = [b[1][1:] if b[1][0] == SOT else b[1] for b in batch]   # model tự thêm decoder_start
    L = max(map(len, ids))
    labels = torch.full((len(ids), L), -100, dtype=torch.long)
    for i, s in enumerate(ids): labels[i, :len(s)] = torch.tensor(s)
    return feats, labels

train_ds = UttDS(train_utts, AUG_P)
print("ví dụ nhãn:", tok.decode(train_ds[0][1]))

In [ ]:
# ===== Dựng model train: điểm xuất phát (đã merge) + LoRA mới =====
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
base = load_start(torch.float32)
base.config.use_cache = False
if SPEC_AUG:
    base.config.apply_spec_augment = True
    base.config.mask_time_prob, base.config.mask_time_length = 0.05, 10
    base.config.mask_feature_prob, base.config.mask_feature_length = 0.05, 10
base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model = get_peft_model(base, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                        bias="none", target_modules=LORA_TARGETS))
model.to(DEV)
model.print_trainable_parameters()

dl = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2,
                                 collate_fn=collate, drop_last=True,
                                 generator=torch.Generator().manual_seed(SEED))
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
epochs = 1 if SMOKE else EPOCHS
total = max(1, epochs * len(dl) // GRAD_ACC)
sched = transformers.get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)
scaler = torch.amp.GradScaler("cuda")
print("optimizer steps:", total, "| steps/epoch:", len(dl))

In [ ]:
# ===== Train: eval dev mỗi epoch, giữ checkpoint ít câu sai nhất, early stop =====
best_m, best_ep, bad = RESULTS[START], -1, 0     # chỉ lưu khi tốt hơn điểm xuất phát
history = []
LORA_PATH = f"{OUT}/best_lora.pt"
if RESUME:
    history = json.load(open(f"{RESUME_DIR}/history.json", encoding="utf-8"))
    json.dump(history, open(f"{OUT}/history.json", "w"), indent=1, ensure_ascii=False)
    for h in history:
        if better(h, best_m): best_m, best_ep = h, h["epoch"]
    LORA_PATH = f"{RESUME_DIR}/best_lora.pt"
for ep in range(0 if RESUME else epochs):
    train_ds.epoch = ep
    model.train(); t0 = time.time(); run = 0.0
    for step, (x, y) in enumerate(dl):
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(input_features=x.to(DEV), labels=y.to(DEV)).loss / GRAD_ACC
        scaler.scale(loss).backward()
        run += loss.item() * GRAD_ACC
        if (step + 1) % GRAD_ACC == 0 or step + 1 == len(dl):
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        if (step + 1) % 100 == 0:
            print(f"ep {ep} step {step+1}/{len(dl)} loss {run/(step+1):.4f} {time.time()-t0:.0f}s")
    model.config.use_cache = True
    m_ep = evaluate(f"ep{ep}", transcribe(model, dev_utts, desc=f"dev ep{ep}"), show=10)
    model.config.use_cache = False
    history.append({"epoch": ep, "train_loss": run / max(1, len(dl)), "sec": time.time() - t0, **m_ep})
    json.dump(history, open(f"{OUT}/history.json", "w"), indent=1, ensure_ascii=False)
    torch.save(get_peft_model_state_dict(model), f"{OUT}/last_lora.pt")
    if better(m_ep, best_m):
        best_m, best_ep, bad = m_ep, ep, 0
        torch.save(get_peft_model_state_dict(model), f"{OUT}/best_lora.pt")
        print(f"  -> best: n_wrong {best_m['n_wrong']} WER {best_m['WER']:.4f}")
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f"early stop sau epoch {ep}"); break
print("best epoch:", best_ep, "| n_wrong", best_m["n_wrong"])

In [ ]:
# ===== Model cuối: greedy vs beam, merge fp16, kiểm tra merge, latency =====
import gc
del opt, sched, scaler, dl, params                 # giải phóng bộ nhớ train trước khi beam search
gc.collect(); torch.cuda.empty_cache()
if best_ep < 0:
    print("Fine-tune không tốt hơn điểm xuất phát -> model cuối = điểm xuất phát (không có LoRA mới).")
    del model; gc.collect(); torch.cuda.empty_cache()
    final = load_start(torch.float16).to(DEV)
else:
    set_peft_model_state_dict(model, torch.load(LORA_PATH, map_location="cpu"))
    final = model.merge_and_unload().half().to(DEV)
final.config.use_cache = True
final.config.apply_spec_augment = False      # trả về hành vi inference như PhoWhisper gốc

beam_res = {}
for b in FINAL_BEAMS:
    # KV cache cross-attention tăng theo batch × beams: giữ batch × beams ≈ EVAL_BATCH để không OOM
    hyps = transcribe(final, dev_utts, bs=max(1, EVAL_BATCH // b), beams=b, desc=f"final beams={b}")
    beam_res[b] = (evaluate(f"final_beam{b}", hyps, show=0), hyps)
best_beam = min(beam_res, key=lambda b: (beam_res[b][0]["n_wrong"], beam_res[b][0]["WER"]))
RESULTS["final"] = {**beam_res[best_beam][0], "beams": best_beam, "start": START, "best_epoch": best_ep}
for b in beam_res: RESULTS[f"final_beam{b}"] = beam_res[b][0]
print("beam chọn:", best_beam)

save_dir = f"{OUT}/phowhisper_medium_ft"
final.generation_config.num_beams = best_beam
final.save_pretrained(save_dir, safe_serialization=True); processor.save_pretrained(save_dir)
print("saved", save_dir, os.listdir(save_dir))

# nạp lại từ đĩa, chép lại 100 câu dev -> phải trùng với model trong RAM
del final; torch.cuda.empty_cache()
reload = WhisperForConditionalGeneration.from_pretrained(save_dir).half().to(DEV)
a = transcribe(reload, dev_utts[:100], bs=max(1, EVAL_BATCH // best_beam), beams=best_beam, desc="reload check")
same = sum(x == y for x, y in zip(a, beam_res[best_beam][1][:100]))
print(f"reload == in-memory: {same}/{len(a)}")
RESULTS["reload_match"] = f"{same}/{len(a)}"
t0 = time.time(); transcribe(reload, dev_utts[:50], bs=1, beams=best_beam, desc="latency bs=1")
RESULTS["latency_ms_per_utt_bs1"] = 1000 * (time.time() - t0) / min(50, len(dev_utts))
json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=1, ensure_ascii=False)

In [ ]:
# ===== Báo cáo: trước/sau + phân tích câu sai =====
import pandas as pd
keys = ["n", "n_wrong", "n_wrong_final", "WER", "CER", "WER_final_turn", "num_seq_acc", "neg_recall"]
tab = pd.DataFrame({k: {c: v.get(c) for c in keys} for k, v in RESULTS.items() if isinstance(v, dict)}).T
print(tab.to_string())
tab.to_csv(f"{OUT}/summary.csv")

df = pd.read_csv(f"{OUT}/dev_final_beam{RESULTS['final']['beams']}.csv").fillna("")
bad = df[df.ok == 0].copy()
def kind(r):
    rw, hw = r.ref_norm.split(), r.hyp_norm.split()
    if [w for w in rw if w in NUM_WORDS] != [w for w in hw if w in NUM_WORDS]: return "số"
    if any(rw.count(k) != hw.count(k) for k in NEG_WORDS): return "phủ định"
    if len(rw) != len(hw): return "thêm/thiếu từ"
    return "thay từ"
bad["loai"] = bad.apply(kind, axis=1) if len(bad) else []
bad.to_csv(f"{OUT}/wrong_final.csv", index=False)
print(f"\n{len(bad)} câu sai / {len(df)} câu dev. Phân loại:\n{bad['loai'].value_counts().to_string() if len(bad) else '-'}")
for _, r in bad.iterrows():
    print(f"[{r.loai}] {r.sid}/{r.turn}\n   REF: {r.ref_norm}\n   HYP: {r.hyp_norm}")

# Đóng gói để tải về / tạo Kaggle Model
shutil.make_archive("/kaggle/working/phowhisper_medium_ft" if OUT.startswith("/kaggle") else f"{OUT}/phowhisper_medium_ft",
                    "zip", f"{OUT}/phowhisper_medium_ft")
print("done")